# 06 Job Structured Extraction

This notebook extracts structured information from an IT job posting.

Main goals:
- load a cleaned IT job posting from the processed dataset
- create a job advertisement text input
- use OpenAI `gpt-4o-mini` through LangChain
- define a structured Pydantic schema for job data
- extract required skills, nice-to-have skills, tools, responsibilities and requirements
- save the structured job posting as JSON

This step does not compare the job posting with a CV.

## 1. Imports and Settings

In [1]:
import os
import json
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv
from typing import List, Optional
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [2]:
load_dotenv(dotenv_path=Path("../.env"))
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
PROCESSED_JOBS_DIR = Path("../data/processed/jobs")
OUTPUT_JOB_EXTRACTION_DIR = Path("../outputs/job_extraction")

In [4]:
cleaned_jobs_path = PROCESSED_JOBS_DIR / "cleaned_it_job_postings.csv"
structured_job_output_path = OUTPUT_JOB_EXTRACTION_DIR / "structured_job.json"

## 2. Load Cleaned IT Job Postings Dataset

In [5]:
df_jobs = pd.read_csv(cleaned_jobs_path)

In [6]:
df_jobs.shape

(8434, 16)

In [7]:
df_jobs.head()

,title,description,company_name,formatted_work_type,work_type,formatted_experience_level,listed_time,original_listed_time,expiry,job_posting_url,clean_job_title,clean_job_description,combined_text,title_has_it_keyword,tech_keyword_count,it_job_category
0,Software Engineer,Job Description:GOYT is seeking a skilled and ...,GOYT,Part-time,PART_TIME,Unknown,1.713281e+12,1.713281e+12,1.715873e+12,https://www.linkedin.com/jobs/view/175485704/?...,Software Engineer,Job Description:GOYT is seeking a skilled and ...,software engineer job description:goyt is seek...,True,6,Frontend Development
1,Full Stack Engineer,"Location: Remote\nCompany Overview:SkillFit, a...",Ideando Inc,Full-time,FULL_TIME,Unknown,1.713493e+12,1.713493e+12,1.716085e+12,https://www.linkedin.com/jobs/view/2234533717/...,Full Stack Engineer,"Location: Remote Company Overview:SkillFit, a ...",full stack engineer location: remote company o...,False,17,Frontend Development
2,Salesforce Vlocity Developer,Role: Salesforce Vlocity DeveloperLocation: Ne...,SysMind,Contract,CONTRACT,Unknown,1.713211e+12,1.713211e+12,1.715803e+12,https://www.linkedin.com/jobs/view/3169712432/...,Salesforce Vlocity Developer,Role: Salesforce Vlocity DeveloperLocation: Ne...,salesforce vlocity developer role: salesforce ...,False,7,DevOps / Cloud
3,Data Architect,Request: Data ArchitectLocation: San Francisco...,Saxon AI,Contract,CONTRACT,Unknown,1.713537e+12,1.713537e+12,1.716129e+12,https://www.linkedin.com/jobs/view/3245063922/...,Data Architect,Request: Data ArchitectLocation: San Francisco...,data architect request: data architectlocation...,False,6,Backend Development
4,Senior Developer,This individual will work with a high performa...,USLI,Full-time,FULL_TIME,Associate,1.713538e+12,1.713538e+12,1.716130e+12,https://www.linkedin.com/jobs/view/3475933396/...,Senior Developer,This individual will work with a high performa...,senior developer this individual will work wit...,False,9,Frontend Development


## 3. Select One Job Posting for Extraction

In [8]:
job_row_index = 1
selected_job = df_jobs.iloc[job_row_index]

In [9]:
selected_job

title                                                       Full Stack Engineer
description                   Location: Remote\nCompany Overview:SkillFit, a...
company_name                                                        Ideando Inc
formatted_work_type                                                   Full-time
work_type                                                             FULL_TIME
formatted_experience_level                                              Unknown
listed_time                                                     1713493353000.0
original_listed_time                                            1713493353000.0
expiry                                                          1716085353000.0
job_posting_url               https://www.linkedin.com/jobs/view/2234533717/...
clean_job_title                                             Full Stack Engineer
clean_job_description         Location: Remote Company Overview:SkillFit, a ...
combined_text                 full stack

In [10]:
selected_job['description']

"Location: Remote\nCompany Overview:SkillFit, a subsidiary of Ideando Inc. is a cutting-edge AI-powered platform that revolutionizes the talent acquisition process by connecting job seekers with recruiters in a seamless and efficient manner. Our mission is to leverage technology to enhance the job search experience for both candidates and employers, ultimately driving better outcomes and reducing frustration in the hiring process.\nPosition Overview:We are seeking a talented and experienced Freelance Full-Stack Software Engineer to collaborate with our dynamic team on a project basis. The ideal candidate will have a passion for building innovative software solutions, a strong foundation in both front-end and back-end technologies, and a desire to contribute to a fast-paced startup environment.\nKey Responsibilities:1. Collaborate with the product team to understand requirements, design solutions, and implement new features and functionalities for SkillFit's platform.2. Develop robust, 

## 4. Define Structured Job Schema

In [11]:
class ExperienceRequirement(BaseModel):
    minimum_years: Optional[str] = Field(
        default=None,
        description="Minimum years of experience required, if explicitly mentioned."
    )

    seniority_level: Optional[str] = Field(
        default=None,
        description="Seniority level such as Junior, Mid, Senior, Lead, if visible or clearly inferable from the job title or description."
    )

    experience_description: Optional[str] = Field(
        default=None,
        description="Short description of the required experience."
    )

In [12]:
class EducationRequirement(BaseModel):
    degree_required: Optional[str] = Field(
        default=None,
        description="Required degree or education level, if mentioned."
    )

    field_of_study: Optional[str] = Field(
        default=None,
        description="Required or preferred field of study, if mentioned."
    )

    is_required: Optional[bool] = Field(
        default=None,
        description="True if education is required, False if it is only preferred, None if unclear."
    )

In [13]:
class StructuredJobPosting(BaseModel):
    job_title: Optional[str] = Field(
        default=None,
        description="Job title extracted from the posting."
    )

    company_name: Optional[str] = Field(
        default=None,
        description="Company name, if visible in the job posting."
    )

    location: Optional[str] = Field(
        default=None,
        description="Job location, if visible."
    )

    work_mode: Optional[str] = Field(
        default=None,
        description="Work mode such as remote, hybrid, on-site or unknown."
    )

    employment_type: Optional[str] = Field(
        default=None,
        description="Employment type such as full-time, part-time, contract, internship or unknown."
    )

    job_category: Optional[str] = Field(
        default=None,
        description="Broad IT job category, such as Frontend Development, Backend Development, Data Analytics, DevOps, QA, Cybersecurity, etc."
    )

    required_skills: List[str] = Field(
        default_factory=list,
        description="Skills that are clearly required in the job posting."
    )

    nice_to_have_skills: List[str] = Field(
        default_factory=list,
        description="Skills that are listed as preferred, optional or nice-to-have."
    )

    programming_languages: List[str] = Field(
        default_factory=list,
        description="Programming languages mentioned in the job posting."
    )

    frameworks_and_libraries: List[str] = Field(
        default_factory=list,
        description="Frameworks and libraries mentioned in the job posting."
    )

    databases: List[str] = Field(
        default_factory=list,
        description="Databases mentioned in the job posting."
    )

    cloud_and_devops_tools: List[str] = Field(
        default_factory=list,
        description="Cloud platforms, DevOps tools and infrastructure tools mentioned in the job posting."
    )

    data_and_ai_tools: List[str] = Field(
        default_factory=list,
        description="Data analysis, BI, machine learning or AI tools mentioned in the job posting."
    )

    testing_tools: List[str] = Field(
        default_factory=list,
        description="Testing or QA tools mentioned in the job posting."
    )

    other_tools: List[str] = Field(
        default_factory=list,
        description="Other software tools mentioned in the job posting."
    )

    responsibilities: List[str] = Field(
        default_factory=list,
        description="Main responsibilities and tasks described in the job posting."
    )

    experience_requirement: ExperienceRequirement = Field(
        default_factory=ExperienceRequirement,
        description="Experience and seniority requirements."
    )

    education_requirement: EducationRequirement = Field(
        default_factory=EducationRequirement,
        description="Education requirements."
    )

    certifications: List[str] = Field(
        default_factory=list,
        description="Required or preferred certifications mentioned in the posting."
    )

    language_requirements: List[str] = Field(
        default_factory=list,
        description="Human language requirements mentioned in the job posting."
    )

    soft_skills: List[str] = Field(
        default_factory=list,
        description="Soft skills mentioned in the job posting."
    )

    unclear_or_missing_information: List[str] = Field(
        default_factory=list,
        description="Important information that is missing, unclear or not explicitly stated in the job posting."
    )

## 5. Initialize OpenAI Model

In [14]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [15]:
structured_llm = llm.with_structured_output(StructuredJobPosting)

## 6. Create Job Structured Extraction Prompt

In [16]:
job_extraction_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an AI assistant specialized in structured extraction from IT job postings.

Your task is to extract structured information from the provided job advertisement text.

Important rules:
- Extract only information that is explicitly present in the job posting.
- Do not invent required skills, tools, experience, education, certifications or responsibilities.
- Do not assume a requirement if it is not visible in the text.
- Separate required skills from nice-to-have skills whenever possible.
- If the job posting uses words such as "required", "must have", "need", or "minimum", classify the item as required.
- If the job posting uses words such as "preferred", "nice to have", "plus", or "advantage", classify the item as nice-to-have.
- If it is unclear whether a skill is required or optional, place it in required_skills only if the text strongly suggests it is necessary.
- Focus on IT-related information: programming languages, frameworks, databases, cloud tools, DevOps tools, testing tools, data tools and software tools.
- Do not compare the job posting with a CV in this step.
- Do not generate recommendations in this step.

Return the result using the required structured schema.
"""
        ),
        (
            "human",
            """
Extract structured information from the following IT job posting:

{job_text}
"""
        )
    ]
)

## 7. Run Structured Job Extraction

In [18]:
job_text=selected_job['clean_job_description']

In [19]:
job_extraction_chain = job_extraction_prompt | structured_llm

structured_job_result = job_extraction_chain.invoke(
    {
        "job_text": job_text
    }
)

In [20]:
structured_job_result

StructuredJobPosting(job_title='Freelance Full-Stack Software Engineer', company_name='SkillFit', location='Remote', work_mode='Remote', employment_type='Freelance', job_category='Software Development', required_skills=['JavaScript', 'TypeScript', 'React', 'Node.js', 'Python', 'Express.js', 'PostgreSQL', 'MongoDB', 'HTML5', 'CSS3', 'AWS', 'Azure', 'Google Cloud Platform', 'version control', 'testing', 'CI/CD'], nice_to_have_skills=['none'], programming_languages=['JavaScript', 'TypeScript', 'Python'], frameworks_and_libraries=['React', 'Node.js', 'Express.js'], databases=['PostgreSQL', 'MongoDB'], cloud_and_devops_tools=['AWS', 'Azure', 'Google Cloud Platform'], data_and_ai_tools=[], testing_tools=[], other_tools=[], responsibilities=["Collaborate with the product team to understand requirements, design solutions, and implement new features and functionalities for SkillFit's platform.", 'Develop robust, scalable, and maintainable code across the full stack, including front-end UI compo

## 8. Convert Structured Result to Dictionary

In [21]:
if hasattr(structured_job_result, "model_dump"):
    structured_job_dict = structured_job_result.model_dump()
else:
    structured_job_dict = structured_job_result.dict()

structured_job_dict

{'job_title': 'Freelance Full-Stack Software Engineer',
 'company_name': 'SkillFit',
 'location': 'Remote',
 'work_mode': 'Remote',
 'employment_type': 'Freelance',
 'job_category': 'Software Development',
 'required_skills': ['JavaScript',
  'TypeScript',
  'React',
  'Node.js',
  'Python',
  'Express.js',
  'PostgreSQL',
  'MongoDB',
  'HTML5',
  'CSS3',
  'AWS',
  'Azure',
  'Google Cloud Platform',
  'version control',
  'testing',
  'CI/CD'],
 'nice_to_have_skills': ['none'],
 'programming_languages': ['JavaScript', 'TypeScript', 'Python'],
 'frameworks_and_libraries': ['React', 'Node.js', 'Express.js'],
 'databases': ['PostgreSQL', 'MongoDB'],
 'cloud_and_devops_tools': ['AWS', 'Azure', 'Google Cloud Platform'],
 'data_and_ai_tools': [],
 'testing_tools': [],
 'other_tools': [],
 'responsibilities': ["Collaborate with the product team to understand requirements, design solutions, and implement new features and functionalities for SkillFit's platform.",
  'Develop robust, scalable

## 9. Preview Extracted Job Profile

In [22]:
job_profile = {
    "job_title": structured_job_dict.get("job_title"),
    "company_name": structured_job_dict.get("company_name"),
    "location": structured_job_dict.get("location"),
    "work_mode": structured_job_dict.get("work_mode"),
    "employment_type": structured_job_dict.get("employment_type"),
    "job_category": structured_job_dict.get("job_category"),
}

In [23]:
job_profile

{'job_title': 'Freelance Full-Stack Software Engineer',
 'company_name': 'SkillFit',
 'location': 'Remote',
 'work_mode': 'Remote',
 'employment_type': 'Freelance',
 'job_category': 'Software Development'}

## 10. Preview Extracted Skills and Tools

In [24]:
skills_preview = {
    "required_skills": structured_job_dict.get("required_skills", []),
    "nice_to_have_skills": structured_job_dict.get("nice_to_have_skills", []),
    "programming_languages": structured_job_dict.get("programming_languages", []),
    "frameworks_and_libraries": structured_job_dict.get("frameworks_and_libraries", []),
    "databases": structured_job_dict.get("databases", []),
    "cloud_and_devops_tools": structured_job_dict.get("cloud_and_devops_tools", []),
    "data_and_ai_tools": structured_job_dict.get("data_and_ai_tools", []),
    "testing_tools": structured_job_dict.get("testing_tools", []),
    "other_tools": structured_job_dict.get("other_tools", []),
}

In [25]:
skills_preview

{'required_skills': ['JavaScript',
  'TypeScript',
  'React',
  'Node.js',
  'Python',
  'Express.js',
  'PostgreSQL',
  'MongoDB',
  'HTML5',
  'CSS3',
  'AWS',
  'Azure',
  'Google Cloud Platform',
  'version control',
  'testing',
  'CI/CD'],
 'nice_to_have_skills': ['none'],
 'programming_languages': ['JavaScript', 'TypeScript', 'Python'],
 'frameworks_and_libraries': ['React', 'Node.js', 'Express.js'],
 'databases': ['PostgreSQL', 'MongoDB'],
 'cloud_and_devops_tools': ['AWS', 'Azure', 'Google Cloud Platform'],
 'data_and_ai_tools': [],
 'testing_tools': [],
 'other_tools': []}

## 11. Convert Extracted Skills to DataFrame

In [26]:
skill_rows = []

skill_categories = [
    "required_skills",
    "nice_to_have_skills",
    "programming_languages",
    "frameworks_and_libraries",
    "databases",
    "cloud_and_devops_tools",
    "data_and_ai_tools",
    "testing_tools",
    "other_tools",
    "soft_skills",
    "language_requirements"
]
for category in skill_categories:
    for item in structured_job_dict.get(category, []):
        skill_rows.append(
            {
                "category": category,
                "item": item
            }
        )

job_skills_df = pd.DataFrame(skill_rows)


In [27]:
job_skills_df

,category,item
0,required_skills,JavaScript
1,required_skills,TypeScript
2,required_skills,React
3,required_skills,Node.js
4,required_skills,Python
5,required_skills,Express.js
6,required_skills,PostgreSQL
7,required_skills,MongoDB
8,required_skills,HTML5
9,required_skills,CSS3


## 12. Preview

In [28]:
responsibilities_df = pd.DataFrame({
    "responsibility": structured_job_dict.get("responsibilities", [])
})

In [29]:
responsibilities_df

,responsibility
0,Collaborate with the product team to understan...
1,"Develop robust, scalable, and maintainable cod..."
2,"Participate in code reviews, architectural dis..."
3,"Work closely with cross-functional teams, incl..."
4,Identify and address technical challenges and ...
5,Contribute to a collaborative and inclusive te...


In [30]:
structured_job_dict.get("experience_requirement", {})

{'minimum_years': '3+',
 'seniority_level': 'Mid',
 'experience_description': 'Professional experience in software development, with a focus on full-stack web development.'}

In [31]:
structured_job_dict.get("education_requirement", {})

{'degree_required': None, 'field_of_study': None, 'is_required': None}

## 13. Add Source Metadata

In [32]:
structured_job_dict["metadata"] = {
    "source_file": cleaned_jobs_path.name,
    "source_row_index": int(job_row_index),
    "model": "gpt-4o-mini",
    "extraction_type": "structured_job_extraction",
    "notes": "Only information visible in the job posting text should be included."
}
if "job_id" in selected_job.index:
    structured_job_dict["metadata"]["job_id"] = str(selected_job.get("job_id"))

if "it_job_category" in selected_job.index:
    structured_job_dict["metadata"]["original_rule_based_category"] = str(selected_job.get("it_job_category"))


In [33]:
structured_job_dict["metadata"]

{'source_file': 'cleaned_it_job_postings.csv',
 'source_row_index': 1,
 'model': 'gpt-4o-mini',
 'extraction_type': 'structured_job_extraction',
 'notes': 'Only information visible in the job posting text should be included.',
 'original_rule_based_category': 'Frontend Development'}

## 14. Save Structured Job JSON

In [34]:
with open(structured_job_output_path, "w", encoding="utf-8") as file:
    json.dump(structured_job_dict, file, indent=4, ensure_ascii=False)